# DEMO: Git Folders and Declarative Automation Bundles

In Databricks, dashboards are **first-class citizens in version control and CI/CD**. There are three approaches:

| Approach | Complexity | Best For |
| --- | --- | --- |
| **Git Folders** | Low | Solo/small team, simple workflows |
| **Export/Import** | Low | Manual backups, point-in-time restore |
| **Declarative Automation Bundles** | Medium-High | Production CI/CD, multi-environment deployment |

| Databricks |
| --- |
| `.lvdash.json` files (human-readable JSON, diffable) |
| **Declarative Automation Bundles** (included) |
| **Git Folders** (native in workspace) |
| Built-in export/import |

In this demo, we'll:
1. Set up a Git Folder and track dashboard changes
2. Export/Import a dashboard as `.lvdash.json`
3. Understand Declarative Automation Bundles for production deployment

> **Note:** Run the setup cell below to create a dashboard you'll practice version-controlling.

---

In [0]:
%run ./Setup

## The Three Approaches to Dashboard Version Control

### 1. Git Folders (Simplest)
A workspace folder directly linked to a Git repository. Changes are tracked automatically.

### 2. Export/Import (Manual)
Export your dashboard as a `.lvdash.json` file, store it anywhere, re-import when needed.

### 3. Declarative Automation Bundles (Production CI/CD)
Define your entire project (dashboard + compute + permissions + schedule) as code. Deploy across environments.

Let's walk through each one.

---

## Part 1: Git Folders

Git Folders provide native Git integration directly in the workspace. Think of a Git Folder as a dedicated space linked to a remote repository (GitHub, GitLab, Bitbucket, or Azure DevOps).

### Step 1: Create a Git Folder
1. In the left sidebar, click **Workspace**
2. Navigate to your user folder
3. Click the **kebab menu** (⋮) next to your folder
4. Select **Create** > **Git Folder**
5. Enter your repository URL (e.g., `https://github.com/your-org/your-repo.git`)
6. Select the branch (e.g., `main`)
7. Click **Create Git Folder**

The folder clones your repository into the workspace.

### Step 2: Move or create a dashboard in the Git Folder
- You can create a new dashboard directly inside the Git Folder
- Or move an existing dashboard into the Git Folder
- The dashboard is stored as a `.lvdash.json` file

### Step 3: (Optional for demo) Connect Databricks to your Git provider
1. Click your **profile icon** (top-right corner)
2. Select **Settings** (or **User Settings**)
3. Navigate to **Linked accounts** (or **Git integration**)
5. Under Git integration click **Add Git credential**
6. Follow the instructions in the pop-up modal

---

## Step-by-Step: Commit and Push Changes

### Step 1: Make a change
1. Open a dashboard or notebook inside your Git Folder
2. Make an edit (e.g., change a widget title, modify a dataset query)
3. The change auto-saves in the workspace

### Step 2: Open the Git dialog
1. Click the **Git branch** indicator next to the file name (shows current branch)
3. The Git dialog opens

### Step 3: Review changes
1. Click the **Changes** tab
2. You'll see which files have been modified (like `git status`)
3. For `.lvdash.json` files, you can see the JSON diff

### Step 4: Commit and push (optional)
1. Pushing to Git only works if you've setup your Git credentials
1. Write a **commit message** (e.g., "Updated revenue chart to show quarterly breakdown")
2. Click **Commit & Push**
3. Changes are pushed to your remote repository

### Working with branches:
- Click the **branch selector** in the Git dialog to switch branches
- **Best practice:** Never work directly on `main`. Create a feature branch for changes:
  1. Click **Create branch**
  2. Name it (e.g., `feature/add-profit-chart`)
  3. Make changes, commit, push
  4. Create a Pull Request in your Git provider
  5. Merge to main after review

### What Git tracks for dashboards:
- Dashboard definition (`.lvdash.json`) — the full JSON structure
- Dataset queries, widget configs, page layouts

### What Git does NOT track:
- Publishing configuration (credential mode, warehouse)
- Schedules and subscriptions
- Warehouse selection

> For those settings, use Declarative Automation Bundles (Part 3).

---

## Part 2: Export/Import (Manual Version Control)

For teams not using Git Folders, you can manually export and import dashboard definitions.

### Export a dashboard:
1. Open your dashboard (draft view)
2. Click the **kebab menu** (⋮) in the top-right corner
3. Select **File actions -> Export**
4. A `.lvdash.json` file downloads to your computer

### What's in the file:
The `.lvdash.json` file is **human-readable JSON**. It contains:
- Page definitions and layout
- Widget configurations (type, encodings, position)
- Dataset queries (full SQL)
- Filter configurations

You can:
- **Diff** between versions (see what changed)
- **Make bulk text replacements** (rename columns across all widgets)
- **Template** dashboards (parameterize catalog/schema names for different environments)

### Import (replace) a dashboard:
1. Create a new dashboard
2. Click the **kebab menu** (⋮)
3. Select **File actions -> Replace** (or **Replace dashboard**)
4. Upload your `.lvdash.json` file
5. The dashboard updates to match the imported definition

### Use cases:
- **Backup:** Export before making risky changes
- **Template:** Export, search-replace schema names, import into a new environment
- **Migrate:** Move dashboards between workspaces (export from one, import to another)
- **Review:** Store exports in Git for code review (even without Git Folders)

---

## Part 3: Declarative Automation Bundles (Production CI/CD)

For production deployments, **Declarative Automation Bundles** (formerly Databricks Asset Bundles) are the gold standard.

A bundle is an **end-to-end definition of your entire project as code**. Instead of having your dashboard in one place and your settings in another, bundles treat everything as source files:

- Business logic (notebooks, Python files)
- Dashboard definitions (`.lvdash.json`)
- Resource metadata (compute, permissions, schedules)
- Deployment targets (dev, staging, prod)

### Why Bundles?

| Without Bundles | With Bundles |
| --- | --- |
| Manually publish to each environment | `databricks bundle deploy --target prod` |
| Permissions configured by hand | Permissions defined in YAML |
| No audit trail for config changes | Full Git history for everything |
| "Works on my machine" problems | Reproducible across environments |

---

## Bundle Structure and Configuration

A Declarative Automation Bundle lives in a Git repository with this structure:

```
my-project/
├── databricks.yml           # Main bundle configuration
├── dashboards/
│   └── sales_report.lvdash.json   # Dashboard definition
├── notebooks/
│   └── etl_pipeline.py        # Supporting code
└── resources/
    └── jobs.yml               # Job/schedule definitions
```

### Example `databricks.yml`:

```yaml
bundle:
  name: sales-dashboard-project

resources:
  dashboards:
    sales_executive:
      display_name: "Sales Executive Dashboard"
      file_path: ./dashboards/sales_report.lvdash.json
      warehouse_id: ${var.sql_warehouse_id}
      permissions:
        - level: CAN_VIEW
          group_name: executives
        - level: CAN_EDIT
          group_name: analytics_team

targets:
  dev:
    default: true
    workspace:
      host: https://dev-workspace.cloud.databricks.com

  prod:
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    variables:
      sql_warehouse_id: "abc123def456"
```

### Key sections:
- **bundle:** Project name and metadata
- **resources:** What to deploy (dashboards, jobs, pipelines)
- **permissions:** Who can access what (defined as code)
- **targets:** Where to deploy (dev, staging, prod)
- **variables:** Environment-specific values (warehouse IDs, schema names)

---

## Step-by-Step: Create and Deploy a Bundle (Workspace UI)

### Step 1: Create a bundle inside a Git Folder
1. In the workspace, navigate to a **Git Folder** (bundles must live in a Git Folder)
2. Click the **Create** button, then click **Bundle**
   - Alternatively, right-click the Git Folder and select **Create > Bundle**
3. Give the bundle a name: `Sales_Dashboard` (letters, numbers, dashes, and underscores only)
4. Select **Empty project**, then click **Next**

This creates a `databricks.yml` scaffold and a `.gitignore` file inside the Git Folder.

### Step 2: Add a dashboard to the bundle
1. Click the `databricks.yml` file to open the **bundle editor**
2. Click the **deployments icon** in the left side menu (rocket) to open the **Deployments pane**
3. Under **Bundle resources**, click **Add**, then **New dashboard definition**
4. Enter a **Dashboard name** and select a **Warehouse**
5. Click **Add and deploy**

This creates a new empty dashboard and a `*.dashboard.yml` configuration file in the bundle. Open and edit the dashboard directly from the **Bundle resources** list. You can also add existing resources to your bundles the same way just select **Add existing [X]** in the **Add** selection menu.

### Step 3: Deploy to development
1. Open `databricks.yml` to enter the bundle editor
2. Click the **deployments icon**
3. In the **Deployments pane**, select your **target** (e.g., `dev`) from the dropdown
4. Click **Deploy**
5. Review the validation summary in the confirmation dialog, then click **Deploy**

Deployment status appears in the **Project output** window. Deployed resources are listed under **Bundle resources** when complete.

### Step 4: Edit and re-deploy a dashboard
1. From the **Deployments pane**, click the dashboard under **Bundle resources**
2. Click **Edit draft** to open the dashboard editor
3. Make your changes
4. Click **Deploy** — this re-deploys all bundle resources, publishing the dashboard changes
5. Click **View deployed** to verify your changes

### Step 5: Deploy to production
The workspace UI **does** support deploying to a `prod` target — simply select it from the target dropdown in the Deployments pane and click **Deploy**. This deploys using your prod target configuration within the current workspace.

> **Important:** Deploying to a *different* workspace (i.e., where `targets.prod.workspace.host` points to a separate Databricks workspace URL) is **not supported** from the workspace UI. For true cross-workspace promotion, use CI/CD:

1. Commit and push your bundle changes by syncing the Git Folder
2. Trigger a CI/CD pipeline that runs the CLI deploy step:

```yaml
# Example GitHub Actions step
- name: Deploy to production
  run: databricks bundle deploy --target prod
  env:
    DATABRICKS_HOST: ${{ secrets.PROD_HOST }}
    DATABRICKS_TOKEN: ${{ secrets.PROD_TOKEN }}
```

---

## Decision Guide: Which Approach?

| Scenario | Approach | Why |
| --- | --- | --- |
| Solo developer, simple workflows | **Git Folders** | Minimal setup, automatic tracking |
| Team with existing Git workflow | **Export/Import + external Git** | Familiar workflow, human-readable diffs |
| Production deployment across environments | **Declarative Automation Bundles** | Full CI/CD, permissions, compute config |
| Quick backup / point-in-time restore | **Export/Import** | Simple file-based backup |
| Regulated industry (audit requirements) | **Bundles + Git** | Complete audit trail for all config |

### Combining approaches:
Many teams use **Git Folders for development** (easy day-to-day tracking) and **Bundles for deployment** (automated promotion to production). These aren't mutually exclusive.

---

---

## Summary

| Approach | What It Does | Key Commands/Actions |
| --- | --- | --- |
| **Git Folders** | Native Git in workspace; auto-tracks changes | Branch → Edit → Commit & Push |
| **Export/Import** | Manual `.lvdash.json` file management | Kebab menu → Export / Import |
| **Bundles** | Infrastructure-as-code deployment | `bundle validate` → `bundle deploy` |

### What each tracks:

| Aspect | Git Folders | Export/Import | Bundles |
| --- | --- | --- | --- |
| Dashboard definition | ✔️ | ✔️ | ✔️ |
| Widget layouts & queries | ✔️ | ✔️ | ✔️ |
| Warehouse assignment | ✘ | ✘ | ✔️ |
| Permissions | ✘ | ✘ | ✔️ |
| Schedule configuration | ✘ | ✘ | ✔️ |
| Multi-environment deploy | ✘ | ✘ (manual) | ✔️ |

### Key Takeaway

Dashboards in Databricks are **code** (JSON). This means they get all the benefits of code:
- Version history
- Code review
- Automated testing
- CI/CD pipelines
- Environment promotion
- Rollback capability